# Oracle4SD Atlas: from native evidence to economic measurements

This tutorial follows the Template's eight stages. It reconstructs one UMA settlement, checks bounded native samples for eight oracle branches, and inspects the daily measurement product. It does not collect every historical record. Each section states its observation unit and output.

| Section | Output | Main code |
|---|---|---|
| [1. Setup](#1-setup) | Version and environment record | `scripts/bootstrap_environment.py` |
| [2. Sources](#2-sources-and-metadata) | Source, scope and rights tables | `metadata/acquisition_profiles.json` |
| [3. Query](#3-querying) | Raw responses and acquisition receipts | `code/query_data/query.py`, `tutorial_revision/query.py` |
| [4. Inspect raw data](#4-queried-data-inspection) | Receipt and event locators | case source specifications |
| [5. Process](#5-processing) | One linked episode and eight sample checks | `code/process_data/process.py` |
| [6. Inspect tables](#6-processed-data-inspection) | Dictionaries, keys and physical inventory | `metadata/revision/table_catalog.json` |
| [7. Describe](#7-descriptive-analysis) | Accounting, missingness and daily summaries | `code/analyze_data/analyze.py` |
| [8. Validate and export](#8-validation-and-reproduction) | Check matrix and evidence ZIP | `code/technical_validation/validate.py` |

An oracle provides claims to a blockchain program. A receipt records transaction execution; its logs record contract events. A reward is distinct from a return of the participant's principal. Protocol resolution does not independently verify an external fact.

## 1. Setup

Use a CPU runtime. The case uses a managed Python 3.12 environment; no key or wallet is needed for archived replay. The small native samples are bundled. Full-table downloads are separate. See the English acceptance guide in `docs/COLAB_ACCEPTANCE.md` for restart, export and independent-reader instructions.

In [1]:
from pathlib import Path
import csv, hashlib, json, os, platform, shutil, subprocess, sys, tempfile, time
from IPython.display import display, Markdown, FileLink
CODE_REF = "71c0e19d3ae2b47d763243371f1bfb97750db61d"
WORK = Path(tempfile.mkdtemp(prefix="oracle-atlas-"))
def command(args, cwd=None):
    env=dict(os.environ, MPLBACKEND="Agg")
    result=subprocess.run([str(a) for a in args],cwd=cwd,env=env,capture_output=True,text=True)
    with (WORK/"commands.log").open("a") as f:
        f.write(json.dumps([str(a) for a in args])+"\n"+result.stdout+result.stderr+"\n")
    if result.returncode:raise RuntimeError(result.stdout+result.stderr)
    return result.stdout
search=[Path.cwd(),Path.cwd().parent]
REPO=next((p for p in search if (p/"configs/atlas_utc_release.json").exists()),None)
if REPO is None:
    REPO=WORK/"repository"
    command(["git","clone","--quiet","https://github.com/Oracle4CEG/Oracle4SD-Atlas.git",REPO])
    command(["git","checkout","--detach",CODE_REF],REPO)
PYTHON=os.environ.get("ORACLE_NOTEBOOK_PYTHON")
if not PYTHON:
    command([sys.executable,REPO/"scripts/bootstrap_environment.py","--output",WORK/"environment"])
    PYTHON=str(WORK/"environment/scientific/bin/python")
import pandas as pd
def read_json(p):return json.loads(Path(p).read_text())
RUN=WORK/"case";RUN.mkdir()
NATIVE=WORK/"native"
shutil.copytree(REPO/"tutorial_revision",NATIVE,ignore=shutil.ignore_patterns("*.ipynb","__pycache__"))
sys.path.insert(0,str(NATIVE));sys.path.insert(0,str(NATIVE/"processing_code"))
from query import query
from replay import run_branch
profiles=read_json(NATIVE/"metadata/profiles.json")
raw_profiles=read_json(NATIVE/"metadata/raw_profiles.json")
def stage(folder, script, extra=(), run=RUN):
    return command([PYTHON,REPO/"code"/folder/script,"--config",REPO/"configs/project.json","--run-dir",run,*extra],REPO)
identity={"code_commit":command(["git","rev-parse","HEAD"],REPO).strip(),
          "working_tree_modified":bool(command(["git","status","--porcelain"],REPO).strip()),
          "data_release":read_json(REPO/"configs/atlas_utc_release.json"),
          "runtime_kind":"hosted_colab" if "google.colab" in sys.modules else "local_jupyter_kernel",
          "host_python":platform.python_version(),"independent_reader":False}
display(identity)

{'code_commit': '71c0e19d3ae2b47d763243371f1bfb97750db61d',
 'working_tree_modified': False,
 'data_release': {'repo_id': 'Oracle4CEG/OracleEconLab-Oracle-Incentives-v1',
  'repo_type': 'dataset',
  'revision': 'atlas-utc-20261029',
  'prefix': 'atlas/releases/atlas-utc-20261029',
  'physical_tables': 100,
  'logical_products': 10,
  'manifest': 'metadata/revision/file_manifest.csv',
  'products': 'metadata/revision/logical_products.json'},
 'runtime_kind': 'local_jupyter_kernel',
 'host_python': '3.12.7',
 'independent_reader': False}

## 2. Sources and metadata

The first register describes seven case files covering five transactions. The historical full-source register has 64 native tables and six archive metadata entries. Native tables are retained collector outputs, not a complete archive of original RPC responses. The current inventory separately describes 100 physical tables.

The eight branches have different scopes. UMA, Chainlink, Tellor, Flare FTSOv2 and Pyth are detailed panels; Chronicle, RedStone Push and DIA are narrower extensions. The acquisition profiles state chains, implementations, range endpoints, source methods and limitations. Project evidence is distributed under CC BY 4.0; upstream rights are retained in `DATA_LICENSE.md`.

In [2]:
display(pd.read_csv(REPO/"metadata/source_register.csv"))
full_sources=pd.read_csv(REPO/"metadata/full_source_register_20261012.csv")
print("Historical source register:",len(full_sources),"entries")
display(full_sources.head(4))
scope=pd.DataFrame(read_json(REPO/"metadata/acquisition_profiles.json")["profiles"])
display(scope[[c for c in ["protocol","chain","delivery_chain","contracts","start","end","limitations"] if c in scope]])
display(pd.DataFrame(profiles)[["branch","parent_native_table","source_mode","sample_rows","acquisition_scope"]])

,source_id,local_path,version,sha256,bytes,origin_url,license,access,scope,acquisition,checked_on
0,case_spec.json,data/data_source/uma_case/case_spec.json,uma-public-rpc-case-v1,6bc9d141f52398b1f3446a93d7619e4e475535419bffec...,6089,project selection specification,CC-BY-4.0 project evidence; third-party rights...,bundled; no credentials,one fixed UMA case,fixed case selection,2026-09-16
1,raw/ethereum_rpc_snapshot.json,data/data_source/uma_case/raw/ethereum_rpc_sna...,uma-public-rpc-case-v1,2af8dd712384c6b67e1323bdb0a9fde4b3849cf8a99bf2...,29216,https://huggingface.co/datasets/Oracle4CEG/Ora...,CC-BY-4.0 project evidence; third-party rights...,bundled; no credentials,one fixed UMA case,frozen JSON-RPC receipt evidence,2026-09-16
2,raw/polygon_rpc_snapshot.json,data/data_source/uma_case/raw/polygon_rpc_snap...,uma-public-rpc-case-v1,bdf308fce350db604af04f889826c16304081a0197b01f...,99358,https://huggingface.co/datasets/Oracle4CEG/Ora...,CC-BY-4.0 project evidence; third-party rights...,bundled; no credentials,one fixed UMA case,frozen JSON-RPC receipt evidence,2026-09-16
3,source/OptimisticOracleV2.sol,data/data_source/uma_case/source/OptimisticOra...,uma-public-rpc-case-v1,9ec0240cea18958d5a7c24990e1d19a27177b8fc1262d1...,32293,https://raw.githubusercontent.com/UMAprotocol/...,AGPL-3.0-only,bundled; no credentials,one fixed UMA case,pinned verified contract source/ABI,2026-09-16
4,source/OptimisticOracleV2Interface.sol,data/data_source/uma_case/source/OptimisticOra...,uma-public-rpc-case-v1,d6a09bb47ed210dcb28635f3e4ddb652dcc106355876c7...,16138,https://raw.githubusercontent.com/UMAprotocol/...,AGPL-3.0-only,bundled; no credentials,one fixed UMA case,pinned verified contract source/ABI,2026-09-16
5,source/VotingV2.abi.json,data/data_source/uma_case/source/VotingV2.abi....,uma-public-rpc-case-v1,84e7656bf39385b2fb4af555422f1ec36d933889be38ee...,42629,https://eth.blockscout.com/api/v2/smart-contra...,CC-BY-4.0 project evidence; third-party rights...,bundled; no credentials,one fixed UMA case,pinned verified contract source/ABI,2026-09-16
6,source/VotingV2.sol,data/data_source/uma_case/source/VotingV2.sol,uma-public-rpc-case-v1,154377b8e445301888b17b888112645daf9a7acd1d9e35...,59632,https://eth.blockscout.com/api/v2/smart-contra...,AGPL-3.0-only,bundled; no credentials,one fixed UMA case,pinned verified contract source/ABI,2026-09-16


Historical source register: 70 entries


,component,path,revision,sha256,bytes,download_url,source_identity,license,scope
0,native,data/curated/parquet/chainlink_eth_usd_reports...,c7df3127756d831a61c781d0531c155dc8d9c2fb,a8a7163c2aa4fd52a19354324116831a749da37bfc7382...,4953032,https://huggingface.co/datasets/Oracle4CEG/Ora...,historical frozen input; original inventory/so...,project derived/native records subject to upst...,offline reconstruction; not fresh network coll...
1,native,data/curated/parquet/chainlink_event_link_flow...,c7df3127756d831a61c781d0531c155dc8d9c2fb,6bfa2d90b7829374dfc5dd9df86c8685530142c79c6738...,3492033,https://huggingface.co/datasets/Oracle4CEG/Ora...,historical frozen input; original inventory/so...,project derived/native records subject to upst...,offline reconstruction; not fresh network coll...
2,native,data/curated/parquet/chainlink_link_flows.parquet,c7df3127756d831a61c781d0531c155dc8d9c2fb,ec9fcb99c4f4b746ac985a5cb351bf2cffb4a0a5577045...,2876017,https://huggingface.co/datasets/Oracle4CEG/Ora...,historical frozen input; original inventory/so...,project derived/native records subject to upst...,offline reconstruction; not fresh network coll...
3,native,data/curated/parquet/chainlink_staking_v02_eve...,c7df3127756d831a61c781d0531c155dc8d9c2fb,bb2ba7e88efddfd8f51d2a9fee66773717c072d52565b4...,5505766,https://huggingface.co/datasets/Oracle4CEG/Ora...,historical frozen input; original inventory/so...,project derived/native records subject to upst...,offline reconstruction; not fresh network coll...


,protocol,chain,delivery_chain,contracts,start,end,limitations
0,UMA,Ethereum mainnet,Polygon PoS,VotingV2 0x004395edb43efca9885cedad51ec9faf93b...,DVM deploy block 16697232; Polygon earliest ad...,Ethereum 25433938; Polygon 89439336,different version deployments; 3 unmatched lin...
1,Chainlink,Ethereum mainnet,Ethereum mainnet,community 0xbc10f2e862ed4502144c7d632a3459f49d...,staking block 18572190; vault 18621067; contro...,25433938,125 unclassified raw staking logs; no external...
2,Tellor,tellor-1 Cosmos SDK,Tellor Layer,application modules; source commit 943a2709ef0...,genesis height 1; first report height 317,height 19890860 at 2026-06-30T23:59:57.550796Z,13 disputes are observed panel; historical rew...
3,Flare_FTSOv2,Flare chain 14,Flare,RewardManager 0xc8f55c5aa2c752ee285bd872855c74...,reward epochs 228; RewardManager deploy block ...,closed epoch 410; cutoff block 64054789,epoch 411 crosses cutoff; provider conditions ...
4,Pyth,Solana mainnet,Pythnet input caps / Solana OIS,pyti8TM4zRVBjmarcgAPmTNNAXYKJv7WVHrkrm6woLN,history 2024-09-11T19:53:53Z; retained rolling...,history last retained tx 2026-06-30T23:11:29Z;...,49 rolling epochs do not cover whole transacti...
5,Chronicle,Ethereum mainnet,Ethereum,72 validated registered contracts; full addres...,first retained common event 2023-08-28; indivi...,25433938,14 paid challenge rewards; no universal publis...
6,RedStone,Ethereum mainnet,Ethereum,9 adapters + 91 feed contracts in frozen registry,first retained common event 2023-12-07; per-co...,25433938,Pull has no global update log; no inferred pub...
7,DIA,Lasernet chain 1050,Lasernet,staking 0x677Cf1299c367F6cf6F3E1669aCC18Fd059a...,deployment block 3874186,31302665,127 positive rewards among 129 withdrawals; st...


,branch,parent_native_table,source_mode,sample_rows,acquisition_scope
0,UMA,polygon_uma_request_rounds,archived_native_replay,8,Native retained subset; fresh chain re-acquisi...
1,Chainlink,chainlink_staking_v02_events,archived_native_replay,8,Native retained subset; fresh chain re-acquisi...
2,Tellor,tellor_disputes,archived_native_replay,8,Native retained subset; fresh chain re-acquisi...
3,Flare FTSOv2,flare_reward_epochs,archived_native_replay,8,Native retained subset; fresh chain re-acquisi...
4,Pyth,pyth_ois_economic_events,archived_native_replay,8,Native retained subset; fresh chain re-acquisi...
5,Chronicle,chronicle_ethereum_events,archived_native_replay,8,Native retained subset; fresh chain re-acquisi...
6,RedStone Push,redstone_ethereum_push_events,archived_native_replay,8,Native retained subset; fresh chain re-acquisi...
7,DIA,dia_staking_withdrawals,archived_native_replay,8,Native retained subset; fresh chain re-acquisi...


## 3. Querying

| Mode | What is obtained | Boundary |
|---|---|---|
| Bundled case | Seven registered files | No fresh API requests |
| Public download | Two pinned receipt snapshots; five bundled support files | Secondary archive download |
| Live case | Five receipts, chain identity, headers and retained code | Fixed case; may encounter access failures |
| Eight-branch replay | One registered raw response and a native subset per branch | Bounded identity checks |

Archived replay is the default. Set `RUN_NETWORK_EXAMPLES=True` to run the separate public-download and live cases. Live responses must pass locator and chain checks; they need not have the same response-envelope bytes as an archive. The eight-branch commands in the acquisition guide each stop after one registered transaction or height, use a 25-second timeout, and retain an error on failure.

In [3]:
stage("query_data","query.py",["--mode","bundled"])
acquisition=read_json(RUN/"reports/acquisition.json")
assert acquisition["source_transactions"]==5
assert all(f["frozen_hash_matches"] for f in acquisition["files"])
raw_checks=[query(NATIVE,p["branch"],"archived_raw") for p in raw_profiles]
display(pd.DataFrame(raw_checks)[["branch","mode","expected_native_block","locator_verified"]])
RUN_NETWORK_EXAMPLES=False
if RUN_NETWORK_EXAMPLES:
    for mode in ["public-download","live"]:
        destination=WORK/mode;destination.mkdir()
        try:
            stage("query_data","query.py",["--mode",mode],destination)
            report=read_json(destination/"reports/acquisition.json")
            if mode=="public-download":assert all(f["frozen_hash_matches"] for f in report["files"])
            stage("process_data","process.py",run=destination)
            stage("technical_validation","validate.py",run=destination)
            display(report)
        except RuntimeError as error:
            (destination/"failure.txt").write_text(str(error))
            print(mode,"failed; retained diagnostic, no empty replacement")
else:
    print("Network examples not executed in this run. Retained bounded query evidence:")
    display(pd.DataFrame(read_json(REPO/"reports/publication_20261029/bounded_acquisition.json"))[["branch","mode","status","elapsed_seconds"]])

,branch,mode,expected_native_block,locator_verified
0,UMA,archived_raw,30622389,True
1,Chainlink,archived_raw,18688297,True
2,Flare FTSOv2,archived_raw,30726907,True
3,Chronicle,archived_raw,19212137,True
4,RedStone Push,archived_raw,18733900,True
5,DIA,archived_raw,4875792,True
6,Pyth,archived_raw,293360328,True
7,Tellor,archived_raw,8073265,True


Network examples not executed in this run. Retained bounded query evidence:


,branch,mode,status,elapsed_seconds
0,UMA,live,passed,0.347
1,Chainlink,live,passed,0.552
2,Flare FTSOv2,live,passed,0.606
3,Chronicle,live,passed,0.546
4,RedStone Push,live,passed,0.489
5,DIA,live,passed,0.657
6,Pyth,live,passed,10.065
7,Tellor,live,passed,1.467


## 4. Queried-data inspection

`transactionHash` locates the transaction; `blockHash` identifies the block; `logIndex` locates an emitted event within its chain log order. The raw integer block number is hexadecimal in an Ethereum JSON-RPC receipt. A decoded event is shown after processing, with the source transaction retained. Neither receipt success nor a matching locator proves the external claim was true.

In [4]:
snapshot=read_json(RUN/"queried_data/uma_case/raw/polygon_rpc_snapshot.json")
receipt=next(iter(snapshot["receipts"].values()))
preview={k:receipt.get(k) for k in ["transactionHash","blockHash","blockNumber","status"]}
preview["first_log"]={k:receipt["logs"][0].get(k) for k in ["address","logIndex","topics","data"]}
display(preview)
display(Markdown("[Original transaction](https://polygonscan.com/tx/"+receipt["transactionHash"]+")"))

{'transactionHash': '0x6fc39485f25f23d42bcb55baf68633b31aa5aa53a881116ecf1382e6a1245ede',
 'blockHash': '0x0c3431e36122a4e52804555266e97001edc8681d78bac9a853d8ba611565fcba',
 'blockNumber': '0x3cb27ad',
 'status': '0x1',
 'first_log': {'address': '0x4d97dcd97ec945f40cf65f87097ace5ea0476045',
  'logIndex': '0xbf',
  'topics': ['0xab3760c3bd2bb38b5bcf54dc79802ed67338b4cf29f3054ded67ed24661e4177',
   '0x083556be4d40e5ffd968df80eae4f655281340fbf43aa1c586dfa2e9baae5605',
   '0x0000000000000000000000006a9d222616c90fca5754cd1333cfd9b7fb6a4f74',
   '0x993dcb8e6344f1e9d280112112ea962137a2cf44c96e735081d036a32b3ef97b'],
  'data': '0x0000000000000000000000000000000000000000000000000000000000000002'}}

[Original transaction](https://polygonscan.com/tx/0x6fc39485f25f23d42bcb55baf68633b31aa5aa53a881116ecf1382e6a1245ede)

## 5. Processing

| Transformation | Input → output unit | Key or rule |
|---|---|---|
| ABI decoding | Receipt logs → 9 selected events | Chain, transaction and log index |
| Token-flow selection | Transfer logs → 8 USDC.e movements | Token contract and source locator |
| Lifecycle linkage | 5 transactions → 1 episode | OOv2 request and DVM identifier |
| Settlement accounting | Gross payment → principal and gain | Integer token units; principal is separated |
| Native sample replay | Bounded table → clock and identity check | Registered SHA-256 and row count |

The eight sample checks verify retained native inputs; they do not derive the full native history from the single raw response. That boundary is deliberate and appears in every sample profile.

In [5]:
stage("process_data","process.py")
processing=read_json(RUN/"reports/processing.json")
display(processing["counts"])
case_events=pd.read_csv(RUN/"processed_data/source_events.csv")
display(case_events.head(1))
sample_checks=[]
for p in profiles:
    checked=run_branch(NATIVE,p["branch"]);sample_checks.append(checked)
    sample=pd.read_parquet(NATIVE/p["input"])
    cols=[c for c in ["event_time_utc","epoch_start_utc","epoch_end_utc","event_time_status"] if c in sample]
    print(p["branch"],"—",p["parent_native_table"],"—",p["sample_rows"],"retained rows")
    display(sample[cols].head(2))
display(pd.DataFrame(sample_checks))

{'abc_economic_measures': 28,
 'canonical_chain_qc': 30,
 'economic_cashflows': 4,
 'economic_episodes': 1,
 'economic_link_qc': 13,
 'lifecycle_timeline': 5,
 'source_events': 9,
 'token_flows': 8,
 'variable_provenance': 12}

,event,source_contract,contract_role,source_tx,source_block,transaction_index,log_index,block_time,oo_request_id,requester,...,proposed_price_raw,expiration_time,request_hash,dvm_time,stamped_ancillary_data_hex,message_hex,message_hash,disputer,resolved_price_raw,gross_payout_raw
0,RequestPrice,0xee3afe347d5c74317041e2618c49534daf887c24,optimistic_oracle_v2,0x6fc39485f25f23d42bcb55baf68633b31aa5aa53a881...,63645613,36,196,1730232350,0x699c0842ebca1553171853de9853d07f0336ca0909c9...,0x6a9d222616c90fca5754cd1333cfd9b7fb6a4f74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


UMA — polygon_uma_request_rounds — 8 retained rows


,event_time_utc,event_time_status
0,2022-11-04 20:20:30+00:00,native_clock
1,2022-11-14 16:54:40+00:00,native_clock


Chainlink — chainlink_staking_v02_events — 8 retained rows


...

RedStone Push — redstone_ethereum_push_events — 8 retained rows


,event_time_utc,event_time_status
0,2023-12-07 10:44:35+00:00,native_clock
1,2023-12-07 10:49:47+00:00,native_clock


DIA — dia_staking_withdrawals — 8 retained rows


,event_time_utc,event_time_status
0,2025-07-08 10:30:45+00:00,native_clock
1,2025-07-10 16:45:26+00:00,native_clock


,branch,rows,clock_fields,source_mode,source_sha256,parent_sha256,source_unit,missing_event_time,check
0,UMA,8,"[event_time_utc, request_reference_time_utc, e...",archived_native_replay,3c270f24376de163b21b3f9cee975049f7b426585f697b...,cf9cd6fa6c857f61bbb81ede3af8b7afc21fc3427d6eca...,polygon_uma_request_rounds,0,sample_identity_and_clock_schema
1,Chainlink,8,"[event_time_utc, payment_time_utc]",archived_native_replay,079207aca6594f26647524dc6f92c5128aa9e33fbce3a7...,b6443ce749b366536d1bcc0bc8b90147f4748527a1544b...,chainlink_staking_v02_events,8,sample_identity_and_clock_schema
2,Tellor,8,"[event_time_utc, dispute_end_time_utc, report_...",archived_native_replay,b2eb2bbeacb6a0350f33a78b7b24976a59f7f88a4ddd44...,626f150907eb28aa635754f080626173294b4d35a67ff4...,tellor_disputes,0,sample_identity_and_clock_schema
3,Flare FTSOv2,8,"[epoch_start_utc, epoch_end_utc, event_time_utc]",archived_native_replay,9ebd417011cd58fa311c8a88c17b2eb17f9610e62b23f2...,e9bd828503356da4b2fa1112ee1950995dc4de591c789e...,flare_reward_epochs,8,sample_identity_and_clock_schema
4,Pyth,8,"[event_time_utc, payment_time_utc]",archived_native_replay,5e4d30ccbb067954b9e0dc25747c2c37f1f2fafeed26b0...,d8318e1ddfa4c010e2bf3f3cb3c410a5846e2576716247...,pyth_ois_economic_events,0,sample_identity_and_clock_schema
5,Chronicle,8,"[event_time_utc, oracle_timestamp_utc, payment...",archived_native_replay,29cd91d46866bf98af70df105635e799bee826a3021a7f...,ddaa05c697ef73d7c8faf21fb91cdc843ce084d83f42b7...,chronicle_ethereum_events,0,sample_identity_and_clock_schema
6,RedStone Push,8,"[event_time_utc, oracle_timestamp_utc]",archived_native_replay,1d23c2286ccfe41466c07d1743afbbe78f4f43e735fa62...,03ea0ca606691b9d30540a8a4aacd8abc92ba2446fd6c1...,redstone_ethereum_push_events,0,sample_identity_and_clock_schema
7,DIA,8,"[event_time_utc, payment_time_utc]",archived_native_replay,91e78cf990c5434b1b81843c5512cbd02e0fb3459e7d0a...,ac78ae2d0bf41d94f61e284cf81e92c30d40b09356d5c3...,dia_staking_withdrawals,0,sample_identity_and_clock_schema


## 6. Processed-data inspection

The episode dictionary defines all 85 fields. Case child tables describe events, transfers, accounting components and checks for the same episode; their row counts are not additional lifecycles. The single-case join is the retained request/episode identity. Source-event and transfer identities also include chain, transaction and log position.

The current full catalogue contains 100 physical tables, replacing the previous 95-table inventory for this release. Shared layers overlap. Clock companions join their parent records by record identifier; filter the parent branch before aggregation. `event_time_utc` is an observed execution time, epoch bounds are interval support, and `event_time_status` states the basis or absence of an event clock. Missing times remain null. Dictionary definition status distinguishes complete revision rules from native fields requiring further semantic review.

In [6]:
episode=read_json(RUN/"processed_data/economic_episode.json")
dictionary=pd.read_csv(REPO/"metadata/data_dictionary.csv")
assert set(dictionary.field_name)==set(episode)
display(dictionary[["field_name","type_or_enum","unit","missing_policy"]].head(8))
case_tables=read_json(REPO/"metadata/table_dictionary.json")["tables"]
display(pd.DataFrame.from_dict(case_tables,orient="index"))
print("The tenth case artifact is economic_episode.json: the same episode as economic_episodes.csv, not an additional observation.")
catalog=pd.DataFrame(read_json(REPO/"metadata/revision/table_catalog.json"))
assert len(catalog)==100
display(catalog[["path","products","layer","observation_unit","rows"]])
full_dictionary=pd.read_csv(REPO/"metadata/revision/field_dictionary.csv")
display(full_dictionary[full_dictionary.field.isin(["event_time_utc","epoch_start_utc","epoch_end_utc","event_time_status"])][["table_path","field","arrow_type","definition","definition_status"]].head(8))
croissant=read_json(REPO/"metadata/revision/croissant.json")
assert len(croissant["recordSet"])==100
print("Croissant recordsets:",len(croissant["recordSet"]),"; metadata parsing is separate from data validation.")

,field_name,type_or_enum,unit,missing_policy
0,schema_version,"""1.0.0""",categorical/string/boolean as schema specifies,JSON null means unavailable or not identifiabl...
1,episode_id,"""string""",categorical/string/boolean as schema specifies,JSON null means unavailable or not identifiabl...
2,protocol,"""string""",categorical/string/boolean as schema specifies,JSON null means unavailable or not identifiabl...
3,mechanism,"""string""",categorical/string/boolean as schema specifies,JSON null means unavailable or not identifiabl...
4,native_unit_type,"[""string"", ""null""]",categorical/string/boolean as schema specifies,JSON null means unavailable or not identifiabl...
5,observation_unit,"[""request"", ""actor_episode"", ""proposal"", ""disp...",categorical/string/boolean as schema specifies,JSON null means unavailable or not identifiabl...
6,security_chain_namespace,"""string""",categorical/string/boolean as schema specifies,JSON null means unavailable or not identifiabl...
7,security_chain_id,"""string""",categorical/string/boolean as schema specifies,JSON null means unavailable or not identifiabl...


,rows,unit,warning,passed,dictionary,primary_key,identity,amount
abc_economic_measures,28,core A/B/C measure availability,33-variable dictionary includes 5 support vari...,NaN,NaN,NaN,NaN,NaN
canonical_chain_qc,30,receipt/header/log/code consistency check,NaN,boolean; all must pass,NaN,NaN,NaN,NaN
economic_cashflows,4,accounting component of episode,components are not additional lifecycles,NaN,NaN,NaN,NaN,NaN
economic_episodes,1,one lifecycle,NaN,NaN,metadata/data_dictionary.csv,[episode_id],NaN,NaN
economic_link_qc,13,cross-chain or settlement linkage check,NaN,boolean; all must pass,NaN,NaN,NaN,NaN
lifecycle_timeline,5,"request, proposal, dispute, resolution or sett...",NaN,NaN,NaN,NaN,NaN,NaN
source_events,9,decoded Polygon protocol event,NaN,NaN,NaN,NaN,chain + transaction_hash + log_index; see CSV ...,NaN
token_flows,8,USDC.e Transfer event in selected receipts,NaN,NaN,NaN,NaN,NaN,decimal-string integer base units; decimals 6
variable_provenance,12,derived variable and evidence path,NaN,NaN,NaN,NaN,NaN,NaN


The tenth case artifact is economic_episode.json: the same episode as economic_episodes.csv, not an additional observation.


,path,products,layer,observation_unit,rows
0,data/native/chainlink_eth_usd_reports.parquet,[Chainlink],native_utc,native feed log,62575
1,data/native/chainlink_event_link_flow_qc.parquet,[Chainlink],native_utc,staking event and token-flow reconciliation,51372
2,data/native/chainlink_link_flows.parquet,[Chainlink],native_utc,LINK transfer log,77203
3,data/native/chainlink_staking_v02_events.parquet,[Chainlink],native_utc,"staking configuration, accounting or action log",174890
4,data/native/chronicle_ethereum_events.parquet,[Chronicle],native_utc,Scribe/ScribeOptimistic log,140526
...,...,...,...,...,...
95,data/evidence/cross_chain/five_transactions.pa...,"[UMA, Chainlink, Tellor, Flare FTSOv2, Pyth, C...",evidence,table-defined supporting observation; see fiel...,5
96,data/evidence/cross_chain/identity_link.parquet,"[UMA, Chainlink, Tellor, Flare FTSOv2, Pyth, C...",evidence,table-defined supporting observation; see fiel...,1
97,data/sensitivity/epoch_allocation_sensitivity....,[Daily measurements],sensitivity,table-defined supporting observation; see fiel...,824
98,data/sensitivity/uma_request_followup.parquet,[Daily measurements],sensitivity,table-defined supporting observation; see fiel...,42676


,table_path,field,arrow_type,definition,definition_status
11,data/native/chainlink_eth_usd_reports.parquet,event_time_utc,"timestamp[ns, tz=UTC]",Supported native event time in UTC; epoch and ...,defined_by_revision_rule
12,data/native/chainlink_eth_usd_reports.parquet,event_time_status,string,Basis for the UTC clock or the reason no event...,defined_by_revision_rule
28,data/native/chainlink_event_link_flow_qc.parquet,event_time_utc,"timestamp[ns, tz=UTC]",Supported native event time in UTC; epoch and ...,defined_by_revision_rule
29,data/native/chainlink_event_link_flow_qc.parquet,event_time_status,string,Basis for the UTC clock or the reason no event...,defined_by_revision_rule
40,data/native/chainlink_link_flows.parquet,event_time_utc,"timestamp[ns, tz=UTC]",Supported native event time in UTC; epoch and ...,defined_by_revision_rule
41,data/native/chainlink_link_flows.parquet,event_time_status,string,Basis for the UTC clock or the reason no event...,defined_by_revision_rule
78,data/native/chainlink_staking_v02_events.parquet,event_time_utc,"timestamp[ns, tz=UTC]",Supported native event time in UTC; epoch and ...,defined_by_revision_rule
80,data/native/chainlink_staking_v02_events.parquet,event_time_status,string,Basis for the UTC clock or the reason no event...,defined_by_revision_rule


Croissant recordsets: 100 ; metadata parsing is separate from data validation.


## 7. Descriptive analysis

The case payment is 1,000 USDC.e: 750 returned principal plus 250 token gain. The configured request reward is a different quantity. USDC.e has six decimals; gas uses another asset and is not silently deducted.

The daily table is a bounded full-release inspection: 13,269 typed rows, with source and timed-population counts. A row count describes released groups, not the number of independent participants. Missingness below uses one episode per field as its denominator; it is tutorial-only. The paper map names which broader products support manuscript figures and appendices.

In [7]:
stage("analyze_data","analyze.py")
summary=read_json(RUN/"analysis/summary.json")
principal=int(summary["principal_returned_raw"]);gain=int(summary["realized_payoff_raw"]);gross=int(summary["gross_payout_raw"])
assert principal+gain==gross==1000000000
display(pd.DataFrame([{"component":"returned principal","raw_units":principal},{"component":"token gain","raw_units":gain},{"component":"gross payment","raw_units":gross}]))
missing=pd.DataFrame([{"field":k,"missing":int(v is None),"denominator":1,"unit":"episode"} for k,v in episode.items()])
missing.to_csv(RUN/"analysis/missingness_with_denominators.csv",index=False)
display(missing[missing.missing==1])
daily=pd.read_parquet(NATIVE/"data_source/derived/daily_measurements.parquet")
assert len(daily)==13269
display(daily.groupby("branch").agg(rows=("branch","size"),coverage=("coverage_status",lambda s:", ".join(sorted(set(s))))))
display(daily.groupby(["branch","mechanism_version","metric"])[["source_population_count","source_timed_count"]].max())
display(pd.read_csv(REPO/"metadata/revision/paper_data_code_map.csv"))

,component,raw_units
0,returned principal,750000000
1,token gain,250000000
2,gross payment,1000000000


,field,missing,denominator,unit
8,capital_cost_usd,1,1,episode
18,delay_cost_usd,1,1,episode
23,dvm_negative_slash_raw,1,1,episode
24,dvm_positive_redistribution_raw,1,1,episode
25,economic_regret_usd,1,1,episode
35,independent_ground_truth,1,1,episode
38,investigation_cost_usd,1,1,episode
45,principal_slashed_raw,1,1,episode
57,reward_forfeited_raw,1,1,episode
79,usd_conversion_source,1,1,episode


,rows,coverage
branch,,
Chainlink,1232,partial
Chronicle,977,partial
DIA,258,partial
Flare FTSOv2,636,partial
Pyth,1272,partial
RedStone Push,937,partial
Tellor,748,partial
UMA,7209,partial


source_population_count  \
branch        mechanism_version metric                                                  
Chainlink     Staking v0.2      reward_paid                                     29927   
                                staking_actions                                 50772   
Chronicle     Ethereum Scribe   challenge_reward_paid                              14   
                                update_actions                                 138634   
DIA           Lasernet staking  reward_paid                                       129   
                                staking_actions                                   436   
Flare FTSOv2  RewardManager     reward_paid                                  18468675   
Pyth          OIS               reward_paid                                    459083   
                                stake_actions                                  122071   
RedStone Push Ethereum Push     delivery_updates                               614319   
Tellor        tellor-1          report_submissions                           82866640   
                                tip_reward_paid                                 20380   
UMA           Ethereum VotingV2 stake_loss_applied                             139375   
                                stake_redistribution_applied                   114815   
              Polygon OOv2      actions_disputer                                 1843   
                                actions_proposer                               166507   
                                actions_requester                              179944   
                                bond_forfeited                                  41758   
                                principal_returned                              41758   
                                reward_paid                                     41758   

                                                              source_timed_count  
branch        mechanism_version metric                                            
Chainlink     Staking v0.2      reward_paid                                 2454  
                                staking_actions                             3000  
Chronicle     Ethereum Scribe   challenge_reward_paid                         14  
                                update_actions                            138634  
DIA           Lasernet staking  reward_paid                                  129  
                                staking_actions                              436  
Flare FTSOv2  RewardManager     reward_paid                             18468675  
Pyth          OIS               reward_paid                               459083  
                                stake_actions                             122071  
RedStone Push Ethereum Push     delivery_updates                          614319  
Tellor        tellor-1          report_submissions                      82866640  
                                tip_reward_paid                            20380  
UMA           Ethereum VotingV2 stake_loss_applied                         11409  
                                stake_redistribution_applied               11510  
              Polygon OOv2      actions_disputer                            1843  
                                actions_proposer                          166507  
                                actions_requester                         179944  
                                bond_forfeited                             41758  
                                principal_returned                         41758  
                                reward_paid                                41758

,artifact,input,script,paper
0,Ecosystem figure,registered implementation and scope sources,paper/working_revision/figure_source/build_eco...,Introduction / Appendix A
1,Daily gallery,daily_measurements and epoch signing policies,code/revision/build_figures.py,Data overview / Appendix D
2,UMA task,uma_prediction_tasks and feature_availability,code/revision/build_prediction.py,Methods / Appendix F
3,Cross-chain gallery,five_transactions and cashflows,code/revision/build_evidence.py,Technical validation / Appendix G
4,Semantic gallery,object correction ledgers and rule summary,code/revision/build_evidence.py,Technical validation / Appendix E


## 8. Validation and reproduction

Case checks, sample identity checks and full-release validation have different populations. A deterministic rerun compares scientific outputs from the same inputs and code. It does not substitute for another reader's attempt or Google-hosted execution. Source completeness, historical parameter bindings and independent factual truth are not established by these tests.

The export includes only the public case, native sample outputs and runtime records. It excludes credentials and the full dataset. Source rights remain those in `DATA_LICENSE.md`. The English acceptance guide explains how an independent reader should retain interventions and report discrepancies.

In [8]:
stage("technical_validation","validate.py")
validation=read_json(RUN/"reports/validation.json")
assert validation["passed"] and validation["passed_checks"]==28
display(pd.DataFrame(validation["checks"]))
display(pd.read_csv(REPO/"metadata/completion_audit/validation_scope.csv"))
SECOND=WORK/"case-repeat"
command([PYTHON,REPO/"scripts/run_pipeline.py","--output",SECOND],REPO)
assert read_json(RUN/"analysis/summary.json")==read_json(SECOND/"analysis/summary.json")
EVIDENCE=WORK/"evidence";EVIDENCE.mkdir()
shutil.copytree(RUN,EVIDENCE/"case")
shutil.copytree(SECOND,EVIDENCE/"case-repeat")
shutil.copytree(NATIVE/"processed_data",EVIDENCE/"native-sample-results")
shutil.copy2(WORK/"commands.log",EVIDENCE/"commands.log")
(EVIDENCE/"identity.json").write_text(json.dumps(identity,indent=2))
(EVIDENCE/"scientific.freeze.txt").write_text(command([PYTHON,"-m","pip","freeze"]))
inventory=[{"path":str(p.relative_to(EVIDENCE)),"bytes":p.stat().st_size} for p in sorted(EVIDENCE.rglob("*")) if p.is_file()]
display(pd.DataFrame(inventory))
(EVIDENCE/"inventory.json").write_text(json.dumps(inventory,indent=2))
export=shutil.make_archive(str(WORK/"oracle-atlas-evidence"),"zip",EVIDENCE)
display(FileLink(export))
print("Case checks and deterministic repeat passed. Independent-reader acceptance remains separate.")

,check,denominator,passed,scope,unit
0,episode_schema,1,True,85-field case schema; not full common schema,episode
1,frozen_reference_values,1,True,previous reconstruction values; not independen...,case
2,episode_identity,1,True,fixed preregistered request,episode
3,principal_not_profit,1,True,integer equality,settlement
4,asset_precision,1,True,this case only,asset
5,four_oov2_lifecycle_events,1,True,raw-derived case assertion,case assertion
6,adapter_parameters_same_request,1,True,raw-derived case assertion,case assertion
7,proposal_deposit_equals_bond_plus_final_fee,1,True,raw-derived case assertion,case assertion
8,disputer_deposit_equals_bond_plus_final_fee,1,True,raw-derived case assertion,case assertion
9,configured_reward_refunded_and_rolled,1,True,raw-derived case assertion,case assertion


,scope,population,criterion,reference,evidence,not_tested
0,UMA case,one linked episode; five transactions,28 top-level checks,"case specification, raw receipts and settlemen...",reports/publication_20261029/tutorial.executed...,external factual truth; population representat...
1,Native replay,eight bounded archived subsets,"SHA-256, row counts and supported clock schema",tutorial_revision/metadata/profiles.json,reports/publication_20261029/tutorial.executed...,full raw-to-native history; continuous collection
2,Live locators,one transaction or height per branch,native locator matches registered target,tutorial_revision/metadata/raw_profiles.json,reports/publication_20261029/bounded_acquisiti...,historical runtime binding; complete pagination
3,Physical release,100 catalogued tables,"hash, row count and dictionary field coverage",metadata/revision/file_manifest.csv,reports/publication_20261029/release_validatio...,human semantic review of all native fields
4,UTC conversion,retained records with supported clocks,"native precision, interval meaning and keyed c...",metadata/revision/temporal_native,metadata/revision/core_temporal_validation.json,unrecovered block times; zero-filled missing days
5,Independent/hosted reproduction,not yet observed,independent receipt and actual hosted execution,docs/COLAB_ACCEPTANCE.md,pending,local execution does not establish either cond...


,path,bytes
0,case/analysis/cashflow_components.csv,93
1,case/analysis/field_availability.csv,3138
2,case/analysis/missingness_with_denominators.csv,2652
3,case/analysis/summary.json,441
4,case/processed_data/abc_economic_measures.csv,7601
...,...,...
122,native-sample-results/pyth/summary.json,441
123,native-sample-results/redstone_push/summary.json,459
124,native-sample-results/tellor/summary.json,471
125,native-sample-results/uma/summary.json,479


/tmp/oracle-atlas-r1iq80wy/oracle-atlas-evidence.zip

Case checks and deterministic repeat passed. Independent-reader acceptance remains separate.


### Paper, data and acceptance

The current manuscript is `paper/working_revision/main.pdf`; Appendix B includes native-to-UTC examples. The dataset's ten products and fixed revision are recorded in `configs/atlas_utc_release.json`. Use `scripts/load_atlas_product.py CONFIG` to list all associated tables, or add `--output DIRECTORY` to retrieve the product's non-shared tables with hash checks. Shared tables require explicit `--include-shared` and subsequent branch/key filtering.

Cite Oracle4CEG, *Oracle4SD Atlas: eight oracle branches, UMA tasks and daily measurements*, version `atlas-utc-20261029`, Hugging Face (2026), together with the resolved dataset commit. No DOI is assigned. The legacy experimental configuration remains available at the predecessor revision recorded in release metadata.

For a failed checksum, use the pinned input again and preserve the failure. For an unavailable historical endpoint, retain the acquisition error and use archived replay only for its declared scope. Restart the kernel and run all cells after an interrupted execution. Report reproducibility issues through the Oracle4SD-Atlas GitHub repository, including the evidence ZIP and any intervention; do not include credentials.